# NYC Rolling Sales — Data Cleaning Pipeline

**Purpose:** Clean the raw NYC Rolling Sales dataset and engineer meaningful features.  
**Input:** `data/raw/nyc-rolling-sales.csv`  
**Output:** `data/processed/cleaned_data.csv`

In [ ]:
# ── 1. IMPORTS & CONFIGURATION ──────────────────────────────────────────────────
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RAW_PATH = os.path.join("..", "data", "raw", "nyc-rolling-sales.csv")
OUT_PATH = os.path.join("..", "data", "processed", "cleaned_data.csv")

In [ ]:
# ── 2. LOAD RAW DATA ────────────────────────────────────────────────────────────
# The CSV has an unnamed index column (first column), so we skip it.
df = pd.read_csv(RAW_PATH, index_col=0)
print(f"Raw shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── 3. STANDARDIZE COLUMN NAMES ─────────────────────────────────────────────────
# Convert to snake_case, strip whitespace, replace hyphens/spaces with underscores.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[\s\-]+", "_", regex=True)
)
print("Standardized columns:", list(df.columns))

In [ ]:
# ── 4. DROP USELESS / IRRELEVANT COLUMNS ────────────────────────────────────────
# ease_ment → 100% blank (single whitespace in every row — zero information)
# address, apartment_number → too granular / PII-adjacent, not useful for analysis
# block, lot → parcel-level identifiers, not analytically meaningful
drop_cols = ["ease_ment", "address", "apartment_number", "block", "lot"]
# Only drop columns that actually exist (safety)
drop_cols = [c for c in drop_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f"Dropped {len(drop_cols)} columns → new shape: {df.shape}")

In [ ]:
# ── 5. STRIP WHITESPACE FROM ALL STRING COLUMNS ─────────────────────────────────
str_cols = df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

In [ ]:
# ── 6. CONVERT NUMERIC COLUMNS STORED AS STRINGS ────────────────────────────────
# sale_price, land_square_feet, gross_square_feet are stored as strings.
# They contain " - " (dash placeholders) which should become NaN.
numeric_str_cols = ["sale_price", "land_square_feet", "gross_square_feet"]
for col in numeric_str_cols:
    if col in df.columns:
        # Replace dash-only values with NaN, then coerce to numeric
        df[col] = df[col].replace(["-", "- ", " -", " - ", " -  "], np.nan)
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversions done.")
print(df[numeric_str_cols].dtypes)

In [ ]:
# ── 7. CONVERT SALE_DATE TO DATETIME & EXTRACT TEMPORAL FEATURES ─────────────────
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")

# Temporal features
df["sale_year"] = df["sale_date"].dt.year.astype("Int64")
df["sale_month"] = df["sale_date"].dt.month.astype("Int64")
df["sale_day_of_week"] = df["sale_date"].dt.day_name()
df["sale_quarter"] = df["sale_date"].dt.quarter.astype("Int64")

print("Date range:", df["sale_date"].min(), "→", df["sale_date"].max())

In [ ]:
# ── 8. CLEAN YEAR_BUILT ─────────────────────────────────────────────────────────
# 6,970 rows have year_built = 0, which is obviously invalid → replace with NaN.
df["year_built"] = df["year_built"].replace(0, np.nan).astype("Int64")

In [ ]:
# ── 9. MAP BOROUGH CODES TO NAMES ───────────────────────────────────────────────
borough_map = {
    1: "Manhattan",
    2: "Bronx",
    3: "Brooklyn",
    4: "Queens",
    5: "Staten Island",
}
df["borough_name"] = df["borough"].map(borough_map)
print("Borough distribution:\n", df["borough_name"].value_counts())

In [ ]:
# ── 10. CLEAN BUILDING CLASS CATEGORY ────────────────────────────────────────────
# Format: "07 RENTALS - WALKUP APARTMENTS" → split into code + description
df["building_class_code"] = df["building_class_category"].str.extract(r"^(\d+\w?)", expand=False)
df["building_class_description"] = (
    df["building_class_category"]
    .str.replace(r"^\d+\w?\s+", "", regex=True)
    .str.strip()
)

In [ ]:
# ── 11. CLEAN TAX CLASS AT PRESENT ──────────────────────────────────────────────
# Some values are blank (whitespace) → replace with NaN
df["tax_class_at_present"] = df["tax_class_at_present"].replace("", np.nan)

In [ ]:
# ── 12. HANDLE ZERO/MISSING SQUARE FOOTAGE ───────────────────────────────────────
# Zero values in land/gross square feet are not real → treat as missing
for col in ["land_square_feet", "gross_square_feet"]:
    if col in df.columns:
        df.loc[df[col] == 0, col] = np.nan

print("Square footage nulls after cleaning:")
print(df[["land_square_feet", "gross_square_feet"]].isnull().sum())

In [ ]:
# ── 13. HANDLE ZERO TOTAL UNITS ─────────────────────────────────────────────────
# ~19,762 rows have total_units = 0; these are often vacant lots or special parcels.
# We keep them but flag them.
df["has_units"] = (df["total_units"] > 0).astype(int)

In [ ]:
# ── 14. FILTER INVALID SALE PRICES ──────────────────────────────────────────────
# Remove rows where sale_price is NaN (the dash-placeholder rows)
before = len(df)
df = df[df["sale_price"].notna()].copy()
print(f"Dropped {before - len(df)} rows with missing sale_price")

# Remove $0 sales — these are ownership transfers, not market transactions
before = len(df)
df = df[df["sale_price"] > 0].copy()
print(f"Dropped {before - len(df)} rows with $0 sale_price")

# Remove very low prices (≤ $100) — non-arm's-length transfers
before = len(df)
df = df[df["sale_price"] > 100].copy()
print(f"Dropped {before - len(df)} rows with sale_price ≤ $100")

print(f"Shape after price filtering: {df.shape}")

In [ ]:
# ── 15. HANDLE OUTLIERS (SALE PRICE) ────────────────────────────────────────────
# Use IQR-based capping at 1st and 99th percentiles to limit extreme outliers
# while preserving the natural NYC price distribution.
p01 = df["sale_price"].quantile(0.01)
p99 = df["sale_price"].quantile(0.99)
before_outlier = len(df)
df = df[(df["sale_price"] >= p01) & (df["sale_price"] <= p99)].copy()
print(f"Outlier removal: dropped {before_outlier - len(df)} rows outside [{p01:,.0f}, {p99:,.0f}]")

In [ ]:
# ── 16. HANDLE OUTLIERS (GROSS SQUARE FEET) ──────────────────────────────────────
# For rows that have gross_sqft, cap extreme values at 99th percentile
valid_gsf = df["gross_square_feet"].dropna()
if len(valid_gsf) > 0:
    gsf_p99 = valid_gsf.quantile(0.99)
    df.loc[df["gross_square_feet"] > gsf_p99, "gross_square_feet"] = gsf_p99
    print(f"Capped gross_square_feet at {gsf_p99:,.0f}")

In [ ]:
# ── 17. ENGINEER NEW FEATURES ───────────────────────────────────────────────────

# 17a. Price per square foot (safe division — NaN where sqft is missing or zero)
df["price_per_sqft"] = np.where(
    (df["gross_square_feet"].notna()) & (df["gross_square_feet"] > 0),
    df["sale_price"] / df["gross_square_feet"],
    np.nan,
)

# 17b. Price per unit (safe division)
df["price_per_unit"] = np.where(
    df["total_units"] > 0,
    df["sale_price"] / df["total_units"],
    np.nan,
)

# 17c. Building age at time of sale
df["building_age"] = np.where(
    df["year_built"].notna() & df["sale_year"].notna(),
    df["sale_year"] - df["year_built"],
    np.nan,
)
# Negative ages (data errors) → NaN
df.loc[df["building_age"] < 0, "building_age"] = np.nan

# 17d. Is residential flag (residential units > 0)
df["is_residential"] = (df["residential_units"] > 0).astype(int)

# 17e. Property size category based on gross square feet
df["size_category"] = pd.cut(
    df["gross_square_feet"],
    bins=[0, 1000, 2500, 5000, 10000, float("inf")],
    labels=["Small", "Medium", "Mid-Large", "Large", "Very Large"],
    right=True,
)

# 17f. Price bracket for quick segmentation
df["price_bracket"] = pd.cut(
    df["sale_price"],
    bins=[0, 250_000, 500_000, 1_000_000, 2_500_000, 5_000_000, float("inf")],
    labels=["<250K", "250K-500K", "500K-1M", "1M-2.5M", "2.5M-5M", "5M+"],
    right=True,
)

# 17g. Floor area ratio (building density) — gross_sqft / land_sqft
df["floor_area_ratio"] = np.where(
    (df["land_square_feet"].notna()) & (df["land_square_feet"] > 0),
    df["gross_square_feet"] / df["land_square_feet"],
    np.nan,
)

print("New features created ✓")

In [ ]:
# ── 18. REORDER COLUMNS FOR CLEAN OUTPUT ─────────────────────────────────────────
# Group columns logically: identifiers → property details → sale info → engineered
col_order = [
    # Location identifiers
    "borough", "borough_name", "neighborhood", "zip_code",
    # Property classification
    "building_class_category", "building_class_code", "building_class_description",
    "building_class_at_present", "building_class_at_time_of_sale",
    "tax_class_at_present", "tax_class_at_time_of_sale",
    # Property dimensions
    "residential_units", "commercial_units", "total_units",
    "land_square_feet", "gross_square_feet", "year_built",
    # Sale information
    "sale_date", "sale_price",
    # Engineered temporal features
    "sale_year", "sale_month", "sale_quarter", "sale_day_of_week",
    # Engineered analytical features
    "price_per_sqft", "price_per_unit", "building_age",
    "floor_area_ratio",
    # Categorical flags & segments
    "is_residential", "has_units", "size_category", "price_bracket",
]
# Only include columns that exist (safety)
col_order = [c for c in col_order if c in df.columns]
# Add any remaining columns not in the order list
remaining = [c for c in df.columns if c not in col_order]
df = df[col_order + remaining]

In [ ]:
# ── 19. FINAL VALIDATION ────────────────────────────────────────────────────────
print("=" * 60)
print("FINAL DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nDtype breakdown:")
print(df.dtypes.value_counts())
print(f"\nNull counts (top non-zero):")
nulls = df.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))
print(f"\nSale price range: ${df['sale_price'].min():,.0f} — ${df['sale_price'].max():,.0f}")
print(f"Sale price median: ${df['sale_price'].median():,.0f}")
print(f"Date range: {df['sale_date'].min()} → {df['sale_date'].max()}")
print(f"\nSample rows:")
print(df.head(3).T)

In [ ]:
# ── 20. SAVE CLEANED DATASET ────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"\n✅ Cleaned dataset saved to: {OUT_PATH}")
print(f"   Rows: {len(df):,}  |  Columns: {len(df.columns)}")